#TAREA 2
## Leonardo Garcia Muñoz

- Obtener información de algún origen de datos (propio o de API)

El conjunto a utilizar se explico en la tarea anterior, pero aqui adjunto el codigo utilizado para obtener la informacion de la API de Youtube, la cual se bajo y guardo en un csv:

In [ ]:
!pip install google-api-python-client

In [ ]:
from googleapiclient.discovery import build
import pandas as pd
import time

API_KEY = "OMITO ESTA KEY PARA EVITAR ACCIDENTES JEJE"#por seguridad no pondre la KEY porque este repositorio es publico jeje

youtube = build("youtube","v3",developerKey=API_KEY)

channel_ids = [
"UCRZpxmNB22q_jXcLlk5N8Kw",
"UCECJDeK0MNapZbpaOzxrUPA",
"UCDZsyOkn-WTiTwgAvZSQ_cg",
"UCmc3SanlEKAdNyNLWu_D7cg",
"UCB_yB7FEmS2ngojWd3Kp-aw",
"UC_lEiu6917IJz03TnntWUaQ",
"UCCCPCZNChQdGa9EkATeye4g",
"UCba3hpU7EFBSk817y9qZkiA",
"UCW39zufHfsuGgpLviKh297Q",
"UCUWt8CNkTQSGeKI4A5r0RKg",
"UCjI0_mBwTLyhmY-D3s0RrVA",
"UCHi6x_VxTLfgDrSD0nU6lGw",
"UCwXh0iKPlI4hXNntPECFSbg"
]

rows = []

MAX_VIDEOS_PER_CHANNEL = 5
MAX_COMMENTS = 90000

try:

    for channel_id in channel_ids:

        if len(rows) >= MAX_COMMENTS:
            break

        print("Canal:", channel_id)

        search_request = youtube.search().list(
            part="snippet",
            channelId=channel_id,
            maxResults=MAX_VIDEOS_PER_CHANNEL,
            order="date",
            type="video"
        )

        search_response = search_request.execute()

        for video in search_response["items"]:

            if len(rows) >= MAX_COMMENTS:
                break

            video_id = video["id"]["videoId"]
            video_title = video["snippet"]["title"]
            channel_title = video["snippet"]["channelTitle"]

            print("Video:", video_title)

            next_page = None

            while True:

                comment_request = youtube.commentThreads().list(
                    part="snippet,replies",
                    videoId=video_id,
                    maxResults=100,
                    pageToken=next_page,
                    textFormat="plainText"
                )

                comment_response = comment_request.execute()

                for item in comment_response["items"]:

                    if len(rows) >= MAX_COMMENTS:
                        break

                    comment = item["snippet"]["topLevelComment"]["snippet"]

                    rows.append({
                        "channel_id": channel_id,
                        "channel_title": channel_title,
                        "video_id": video_id,
                        "video_title": video_title,
                        "author": comment["authorDisplayName"],
                        "comment": comment["textDisplay"],
                        "likes": comment["likeCount"],
                        "published_at": comment["publishedAt"],
                        "reply_count": item["snippet"]["totalReplyCount"]
                    })

                    if "replies" in item:

                        for reply in item["replies"]["comments"]:

                            if len(rows) >= MAX_COMMENTS:
                                break

                            r = reply["snippet"]

                            rows.append({
                                "channel_id": channel_id,
                                "channel_title": channel_title,
                                "video_id": video_id,
                                "video_title": video_title,
                                "author": r["authorDisplayName"],
                                "comment": r["textDisplay"],
                                "likes": r["likeCount"],
                                "published_at": r["publishedAt"],
                                "reply_count": 0
                            })

                next_page = comment_response.get("nextPageToken")

                if not next_page:
                    break

                time.sleep(0.5)

except Exception as e:
    print("Error detectado:", e)
    print("Guardando dataset parcial...")

finally:
    df = pd.DataFrame(rows)
    df.to_csv("youtube_bigdata_comments_dataset_2.csv", index=False)
    print("Dataset guardado con", len(df), "comentarios")

df = pd.DataFrame(rows)

df.to_csv("youtube_bigdata_comments_dataset_2.csv", index=False)

print("Dataset guardado:", len(df), "comentarios")

###Entonces con esa data ya guardada, solo subo mi csv al ambiente de google collab y ya trabajo con el
(Para esta tarea no convertire el csv a .parquet porque aun tiene un tamaño "soportable" para trabajar con el)

Adicionalmente volvi a correr el codigo para otros canales, para contar con mas volumen de datos, adjunto los channels IDs para los que hice la nueva extraccion:

In [ ]:
channel_ids = [
"UCwXh0iKPlI4hXNntPECFSbg", #Zazza el italiano (vlogs y viajes)
"UCJCen5xxTtU7cnVHe3n424A", #Lethal Crysis (vlogs y viajes)
"UCLyFFksH0lfVFzQZjmh5Ccw", #Relatos forenses (Crimenes e investigacion)
"UCSG0wVzhyjp3xdpY6CfelbA", #HiClavero (vlogs y viajes)
"UCyJbb85EUgWjl38GcF236Jg" #Eva Maria Beristain (vlogs y sociologia)
]

###Recordemos que este tema de correr varias veces distintos grupos de canales es por el limite de consultas que permite YT a su API

- Convertir el origen de datos a RDD con pySpark

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tarea2BIGDATA") \
    .getOrCreate()

In [1]:
from pyspark import SparkContext

In [2]:
sc = SparkContext("local", "RDDs")

In [21]:
df = spark.read.csv(
    "/content/youtube_bigdata_comments_dataset_2.csv",
    header=True,
    inferSchema=False,
    multiLine=True,
    quote='"',
    escape='"',
    mode="PERMISSIVE"
)#aqui leemos el csv

In [22]:
df.printSchema()

root
 |-- channel_id: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- video_title: string (nullable = true)
 |-- author: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- published_at: string (nullable = true)
 |-- reply_count: string (nullable = true)



In [23]:
from pyspark.sql.functions import col

df = df.withColumn("likes", col("likes").cast("float"))#aqui cambiamos el tipo de dato que tiene "likes"

In [24]:
rdd = df.rdd #aqui convertimos literalmente a RDD

- Realizar alguna operación en el RDD, como estadísticas descriptivas básicas

In [26]:
#Usando Funcion map obtenemos el promedio de likes por comentario
likes_rdd = rdd.map(lambda elemento: elemento['likes'])
avg_likes = likes_rdd.mean()
avg_likes

13.428487951013937

In [28]:
#maximo y minimo de likes
max_likes = likes_rdd.max()
min_likes = likes_rdd.min()

print("Numero maximo de likes en un comentario",max_likes)
print("Numero minimo de likes en un comentario",min_likes)

Numero maximo de likes en un comentario 45738.0
Numero minimo de likes en un comentario 0.0


In [29]:
#hacemos un count de registros
print("Total de registros")
rdd.count()

Total de registros


121504

In [33]:
#filtrar comentarios con mas de 500 likes
comentario_popular = rdd.filter(lambda elemento: elemento['likes'] > 500)
comentario_popular.count()#contamos los que si cumplieron con el filter

484